# Finetuning a BERT Model

In this exercise, you will have to create a BERT model and then train it on the CoLA dataset which has a list of sentences that are either grammatically correct or incorrect. The data loading and model training, testing logic are already included in your code. You will need to fetch the data, and use a pretrained BERT model to solve a classification task.

**In this workspace you have GPU to help train the model but it is best practice to DISABLE it while writing code and only ENABLE it when you are training.**

Here are the steps you need to do to complete this exercise:

1. Data has already been downloaded from [here](https://nyu-mll.github.io/CoLA/) into the `cola_public` directory.
2. Create a tokenizer for the BERT Model
3. Create the BERT Model and the optimizer for the model
4. Save all your work and then **ENABLE** the GPU
5. Run the Package Installations.
5. Run the file to make sure that the model is training properly.
6. If it works, remember to **DISABLE** the GPU before moving to the next page. 

In case you get stuck, you can look at the solution by clicking the jupyter symbol at the top left and navigating to `finetune_a_bert_solution.py`.

## Try It Out!
- Can you train a different BERT architecture and compare the results? 
- Can you finetune the same model on a different dataset or task and compare the result?

## Package Installations
**NOTE**: Everytime you start the GPU, run this before your code. 

In [1]:
!pip install pip --upgrade
!pip install datasets==1.8.0
!pip install transformers==4.6.1
!pip install ipywidgets
#!pip list

  Using cached datasets-1.8.0-py3-none-any.whl.metadata (9.3 kB)
  Using cached pyarrow-3.0.0.tar.gz (682 kB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × installing build dependencies for pyarrow did not run successfully.
  │ exit code: 2
  ╰─> [89 lines of output]
      Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
        Using cached cython-3.3.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (28 kB)
        Using cached numpy-1.19.4.zip (7.3 MB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
      ERROR: Exception:
      Traceback (most recent call last):
        File "/opt/conda/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 109, in _run_wrapper
       

In [2]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

## Starter Code

**Remember** to DISABLE the GPU when you are not training.

In [9]:
import json
import os
import sys

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.utils.data
import torch.utils.data.distributed
from torch.utils.data import DataLoader, RandomSampler, TensorDataset, SequentialSampler
from torch.optim import AdamW
from transformers import BertForSequenceClassification, BertTokenizer


MAX_LEN = 64  # this is the max length of the sentence
batch_size=64
epochs=1

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  #TODO: Create the BERT tokenizer

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [10]:
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [11]:
def get_train_data_loader(batch_size):
    dataset = pd.read_csv(os.path.join("data", "train.csv"))
    sentences = dataset.sentence.values
    labels = dataset.label.values

    input_ids = []
    for sent in sentences:
        encoded_sent = tokenizer.encode(sent, add_special_tokens=True)
        input_ids.append(encoded_sent)

    # pad shorter sentences
    input_ids_padded = []
    for i in input_ids:
        while len(i) < MAX_LEN:
            i.append(0)
        input_ids_padded.append(i)
    input_ids = input_ids_padded

    # mask; 0: added, 1: otherwise
    attention_masks = []
    # For each sentence...
    for sent in input_ids:
        att_mask = [int(token_id > 0) for token_id in sent]
        attention_masks.append(att_mask)

    # convert to PyTorch data types.
    train_inputs = torch.tensor(input_ids)
    train_labels = torch.tensor(labels)
    train_masks = torch.tensor(attention_masks)

    train_data = TensorDataset(train_inputs, train_masks, train_labels)
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)

    return train_dataloader

In [12]:
def get_test_data_loader(test_batch_size):
    dataset = pd.read_csv(os.path.join("data", "test.csv"))
    sentences = dataset.sentence.values
    labels = dataset.label.values

    input_ids = []
    for sent in sentences:
        encoded_sent = tokenizer.encode(sent, add_special_tokens=True)
        input_ids.append(encoded_sent)

    # pad shorter sentences
    input_ids_padded = []
    for i in input_ids:
        while len(i) < MAX_LEN:
            i.append(0)
        input_ids_padded.append(i)
    input_ids = input_ids_padded

    # mask; 0: added, 1: otherwise
    attention_masks = []
    # For each sentence...
    for sent in input_ids:
        att_mask = [int(token_id > 0) for token_id in sent]
        attention_masks.append(att_mask)

    # convert to PyTorch data types.
    test_inputs = torch.tensor(input_ids)
    test_labels = torch.tensor(labels)
    test_masks = torch.tensor(attention_masks)

    test_data = TensorDataset(test_inputs, test_masks, test_labels)
    # test_sampler = RandomSampler(test_data)  
    test_sampler = SequentialSampler(test_data)  # Sequential is the right choice for testing, because the results stay in a fixed order and are easier to reproduce
    test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=test_batch_size)

    return test_dataloader


In [13]:
def test(model, test_loader, device):
    model.eval()
    _, eval_accuracy = 0, 0

    with torch.no_grad():
        for batch in test_loader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)

            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
            logits = outputs[0]
            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to("cpu").numpy()  # No .detach() is needed here, because labels never track gradients
            tmp_eval_accuracy = flat_accuracy(logits, label_ids)
            eval_accuracy += tmp_eval_accuracy

    # print("Test set: Accuracy: ", eval_accuracy/len(test_loader.dataset))  # test divides by len(test_loader.dataset) but accumulates per-batch accuracy. That gives a wrong result when the last batch is smaller than the others. The standard pattern is to divide by len(test_loader) (number of batches) if you’re summing per-batch accuracy
    print("Test set: Accuracy: ", eval_accuracy / len(test_loader))

In [14]:
def train(device):
    train_loader = get_train_data_loader(batch_size)
    test_loader = get_test_data_loader(batch_size)

    model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2, output_attentions = False, output_hidden_states = False)  #TODO: Create the BERT Model
    model=model.to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8) #TODO: Create the optimizer

    for epoch in range(1, epochs + 1):
        total_loss = 0
        model.train()
        for step, batch in enumerate(train_loader):
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)
            model.zero_grad()

            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
            loss = outputs[0]

            total_loss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            # modified based on their gradients, the learning rate, etc.
            optimizer.step()
            if step % 10  == 0:
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)] Loss: {:.6f}".format(
                        epoch,
                        step * len(batch[0]),
                        len(train_loader.sampler),
                        100.0 * step / len(train_loader),
                        loss.item(),
                    )
                )

    test(model, test_loader, device)

In [15]:
if __name__ == "__main__":
    df = pd.read_csv(
        "./cola_public/raw/in_domain_train.tsv",
        sep="\t",
        header=None,
        usecols=[1, 3],
        names=["label", "sentence"],
    )
    sentences = df.sentence.values
    labels = df.label.values

    from sklearn.model_selection import train_test_split

    train_df, test_df = train_test_split(df)
    train_df.to_csv("./data/train.csv", index=False)
    test_df.to_csv("./data/test.csv", index=False)
    device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(device)
    train(device)

cuda:0


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Train Epoch: 1 [0/6413 (0%)] Loss: 0.855146
Train Epoch: 1 [640/6413 (10%)] Loss: 0.623081
Train Epoch: 1 [1280/6413 (20%)] Loss: 0.501462
Train Epoch: 1 [1920/6413 (30%)] Loss: 0.534058
Train Epoch: 1 [2560/6413 (40%)] Loss: 0.461184
Train Epoch: 1 [3200/6413 (50%)] Loss: 0.463826
Train Epoch: 1 [3840/6413 (59%)] Loss: 0.513124
Train Epoch: 1 [4480/6413 (69%)] Loss: 0.321598
Train Epoch: 1 [5120/6413 (79%)] Loss: 0.466914
Train Epoch: 1 [5760/6413 (89%)] Loss: 0.423494
Train Epoch: 1 [1300/6413 (99%)] Loss: 0.585615
Test set: Accuracy:  0.7997030542986425
